### Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import init
from kaggledatahandler import KaggleDataHandler
from audiopreprocessing import SoundDS
from dnn_model.dnn import AudioClassifier
import numpy as np
import os
import datetime
import matplotlib.pyplot as plt


### Creating dataset (Training & Test)

In [2]:
KDHandler = KaggleDataHandler()
datasets_filepath_organized, y_datasets_filepath_organized = KDHandler.create_set()
print("Created the following filepaths with correspondong label\n:", datasets_filepath_organized.keys())


Created the following filepaths with correspondong label
: dict_keys(['fold1', 'fold2', 'fold3', 'fold4', 'fold5', 'fold6', 'fold7', 'fold8', 'fold9', 'fold10'])


In [3]:

preprocessed_datasets = {}
audiopreprocessor = SoundDS()

for fold in datasets_filepath_organized:
    new_prepro_fold = []
    for i, file in enumerate(datasets_filepath_organized[fold]):
        class_ID = y_datasets_filepath_organized[fold][i]
        filepath = file
        spectrgram, class_id = audiopreprocessor.__getitem__(filepath, class_ID)
        preprocessed_wav = (spectrgram, class_id)
        new_prepro_fold.append(preprocessed_wav)
    print("Finished", fold)
    preprocessed_datasets[fold] = new_prepro_fold

Finished fold1
Finished fold2
Finished fold3
Finished fold4
Finished fold5
Finished fold6
Finished fold7
Finished fold8
Finished fold9
Finished fold10


### Testing loop

In [ ]:

def test(model, val_dl):
  correct_prediction = 0
  total_prediction = 0

  # Disable gradient updates
  with torch.no_grad():
    for data in val_dl:
      # Get the input features and target labels, and put them on the GPU
      inputs, labels = data[0].to(device), data[1].to(device)

      # Normalize the inputs
      inputs_m, inputs_s = inputs.mean(), inputs.std()
      inputs = (inputs - inputs_m) / inputs_s

      # Get predictions
      outputs = model(inputs)

      # Get the predicted class with the highest score
      _, prediction = torch.max(outputs,1)
      # Count of predictions that matched the target label
      correct_prediction += (prediction == labels).sum().item()
      total_prediction += prediction.shape[0]
    
  acc = correct_prediction/total_prediction
  print(f'Accuracy: {acc:.2f}, Total items: {total_prediction}\n')
  return acc, fold


### Training loop

In [ ]:
def training(model, train_dl, num_epochs, device):
  # Loss Function, Optimizer and Scheduler
  criterion = nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
  scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.001,
                                                steps_per_epoch=int(len(train_dl)),
                                                epochs=num_epochs,
                                                anneal_strategy='linear')
  
  losses = []
  accuracies = []

  # Repeat for each epoch
  for epoch in range(num_epochs):
    running_loss = 0.0
    correct_prediction = 0
    total_prediction = 0

    # Repeat for each batch in the training set
    for i, data in enumerate(train_dl):
        # Get the input features and target labels, and put them on the GPU
        inputs, labels = data[0].to(device), data[1].to(device)

        # Normalize the inputs
        inputs_m, inputs_s = inputs.mean(), inputs.std()
        inputs = (inputs - inputs_m) / inputs_s

        # Zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

        # Keep stats for Loss and Accuracy
        running_loss += loss.item()

        # Get the predicted class with the highest score
        _, prediction = torch.max(outputs,1)
        # Count of predictions that matched the target label
        correct_prediction += (prediction == labels).sum().item()
        total_prediction += prediction.shape[0]

        #if i % 10 == 0:    # print every 10 mini-batches
        #    print('[%d, %5d] loss: %.3f' % (epoch + 1, i + 1, running_loss / 10))
    
    # Print stats at the end of the epoch
    num_batches = len(train_dl)
    avg_loss = running_loss / num_batches
    acc = correct_prediction/total_prediction
    losses.append(avg_loss)
    accuracies.append(acc)

    print(f'Epoch: {epoch}, Loss: {avg_loss:.2f}, Accuracy: {acc:.2f}')

  print('Finished Training\n')
  return losses, accuracies

### Ploting function

In [ ]:
def plot_and_save_cv_curves(
    train_accuracies,
    train_losses,
    test_accuracies,
    test_accuracy_labels=None,
    base_dir="results"
):
    # 1. Create main result directory
    if not os.path.exists(base_dir):
        os.makedirs(base_dir)

    # 2. Create timestamped folder
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    result_path = os.path.join(base_dir, timestamp)
    os.makedirs(result_path)

    # 3. Create training and test subfolders
    train_path = os.path.join(result_path, "training")
    test_path = os.path.join(result_path, "test")
    os.makedirs(train_path)
    os.makedirs(test_path)

    print(f"Saving all plots to: {result_path}")

    # ---------------------------
    # 4. Training curves per fold
    # ---------------------------
    for fold_idx in sorted(train_accuracies.keys()):
        accs = train_accuracies[fold_idx]
        losses = train_losses[fold_idx]
        epochs = list(range(1, len(accs) + 1))

        plt.figure(figsize=(10, 4))

        # Loss curve
        plt.subplot(1, 2, 1)
        plt.plot(epochs, losses, marker='o')
        plt.title(f"Fold {fold_idx + 1} - Training Loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.grid(True)

        # Accuracy curve
        plt.subplot(1, 2, 2)
        plt.plot(epochs, accs, marker='o')
        plt.title(f"Fold {fold_idx + 1} - Training Accuracy")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.grid(True)

        plt.tight_layout()
        plt.savefig(os.path.join(train_path, f"fold_{fold_idx+1}_training_curves.png"))
        plt.close()

    # ----------------------------------------------
    # 5. Test accuracy histogram
    # ----------------------------------------------

    # Extract single accuracy value per fold
    fold_labels = []
    fold_values = []

    for fold_idx in sorted(test_accuracies.keys()):
        # Get label
        if test_accuracy_labels:
            label_list = test_accuracy_labels.get(fold_idx)
            label = label_list[0] if isinstance(label_list, list) else label_list
        else:
            label = f"Fold {fold_idx + 1}"

        fold_labels.append(label)
        fold_values.append(test_accuracies[fold_idx][0])   # first/only accuracy

    plt.figure(figsize=(7, 5))
    plt.bar(fold_labels, fold_values)
    plt.title("Test Accuracy per Fold")
    plt.xlabel("Fold")
    plt.ylabel("Accuracy")
    plt.grid(axis='y')
    plt.tight_layout()

    plt.savefig(os.path.join(test_path, "test_accuracy_histogram.png"))
    plt.close()

    print("Plots saved successfully.")
    return result_path

### Training & Test

In [ ]:
number_of_test_folds = 1
sets = KDHandler.create_splits(number_of_test_folds)
models = []
num_epochs = 5

training_losses_all = {}
training_accuracies_all = {}
test_accuracies_all = {}
test_accuracies_all_fold = {}


for i, combination_set in enumerate(sets):

    # Create the model and put it on the GPU if available
    new_model = AudioClassifier()
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    new_model = new_model.to(device)
    # Check that it is on Cuda
    next(new_model.parameters()).device

    print(f"##### Set {i} ####\n")

    print(f" - Starting training for set #{i}\n")
    train_set =[]
    for fold in combination_set[1]:
        if len(train_set) == 0:
            train_set = preprocessed_datasets[fold]
        else:
            train_set = train_set + preprocessed_datasets[fold]
 
    print(f"Training from the folds {combination_set[1]}\n")
    train_dl = torch.utils.data.DataLoader(train_set, batch_size=16, shuffle=False)
    losses, accuracies = training(new_model, train_dl, num_epochs, device)
    training_losses_all[i] = losses
    training_accuracies_all[i] = accuracies
    
    print(f"Starting testing for set #{i}\n")
    new_test_acc = []
    new_test_acc_fold = []   
    for fold in combination_set[0]:
        print(f"Testing on fold #{fold}")
        test_dl = torch.utils.data.DataLoader(preprocessed_datasets[fold], batch_size=16, shuffle=False)  
        acc, fold = test(new_model, test_dl)
        new_test_acc.append(acc)
        new_test_acc_fold.append(fold)
    test_accuracies_all[i] = new_test_acc
    test_accuracies_all_fold[i] = new_test_acc_fold

    models.append(new_model)
    

##### Set 0 ####

 - Starting training for set #0

Training from the folds ['fold2', 'fold3', 'fold4', 'fold5', 'fold6', 'fold7', 'fold8', 'fold9', 'fold10']

Epoch: 0, Loss: 1.93, Accuracy: 0.32
Epoch: 1, Loss: 1.46, Accuracy: 0.49
Epoch: 2, Loss: 1.24, Accuracy: 0.58
Epoch: 3, Loss: 1.12, Accuracy: 0.62
Epoch: 4, Loss: 1.07, Accuracy: 0.64
Finished Training

Starting testing for set #0

Testing on fold #fold1
Accuracy: 0.54, Total items: 873

##### Set 1 ####

 - Starting training for set #1

Training from the folds ['fold1', 'fold3', 'fold4', 'fold5', 'fold6', 'fold7', 'fold8', 'fold9', 'fold10']

Epoch: 0, Loss: 1.93, Accuracy: 0.33
Epoch: 1, Loss: 1.49, Accuracy: 0.49
Epoch: 2, Loss: 1.26, Accuracy: 0.57
Epoch: 3, Loss: 1.13, Accuracy: 0.62
Epoch: 4, Loss: 1.08, Accuracy: 0.64
Finished Training

Starting testing for set #1

Testing on fold #fold2
Accuracy: 0.53, Total items: 888

##### Set 2 ####

 - Starting training for set #2

Training from the folds ['fold1', 'fold2', 'fold4',

### Results

In [ ]:

plot_and_save_cv_curves(training_accuracies_all,training_losses_all,test_accuracies_all,test_accuracies_all_fold,"results")

Saving all plots to: results/2025-11-26_15-49-16
Plots saved successfully.


'results/2025-11-26_15-49-16'